# Observability — Stats, Latency, and Logging

Medha 0.3.0 ships a built-in stats collector and structured logging so you can
measure cache effectiveness and diagnose misses without any external tooling.

- **`medha.stats()`** — frozen snapshot: hit/miss rates, latency percentiles (p50/p95/p99), per-strategy breakdown
- **`medha.reset_stats()`** — zero all counters (useful between load tests)
- **`Settings(collect_stats=False)`** — disable collection to eliminate the async-lock overhead
- **Python `logging`** — every tier logs at DEBUG; hits/stores log at INFO

**Requirements:** `pip install "medha-archai[fastembed]"`

In [ ]:
from medha import Medha, Settings, SearchStrategy
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

## Cell 1: Basic Stats Snapshot — `await medha.stats()`

`stats()` returns a **frozen** `CacheStats` object. Call it at any point — it
reflects every `search()` call made so far in this instance.

In [ ]:
settings = Settings(backend_type="memory")

async with Medha("obs_basic", embedder=embedder, settings=settings) as medha:
    await medha.store("How many users?",    "SELECT COUNT(*) FROM users")
    await medha.store("List all products",  "SELECT * FROM products")
    await medha.store("Revenue last month", "SELECT SUM(amount) FROM orders WHERE month = LAST_MONTH()")

    await medha.search("How many users?")           # L1 / exact
    await medha.search("Total user count")          # semantic
    await medha.search("Show me all products")      # semantic
    await medha.search("Something unrelated XYZ")   # no_match

    s = await medha.stats()
    print(s)  # compact __str__ summary
    print()
    print(f"hit_rate     : {s.hit_rate:.1%}")
    print(f"miss_rate    : {s.miss_rate:.1%}")
    print(f"avg_latency  : {s.avg_latency_ms:.1f} ms")
    print(f"backend_count: {s.backend_count}")

## Cell 2: Per-Strategy Breakdown — `stats.by_strategy`

`by_strategy` is a `dict[str, StrategyStats]` keyed by strategy name.
Each value exposes `.count`, `.total_latency_ms`, and `.avg_latency_ms`.

In [ ]:
settings = Settings(backend_type="memory")

async with Medha("obs_strategies", embedder=embedder, settings=settings) as medha:
    await medha.store("Count users",     "SELECT COUNT(*) FROM users")
    await medha.store("Active sessions", "SELECT COUNT(*) FROM sessions WHERE active = true")
    await medha.store("Top 10 products", "SELECT * FROM products ORDER BY sales DESC LIMIT 10")

    queries = [
        "Count users",                   # exact → then L1
        "Count users",                   # L1
        "How many users are there?",     # semantic
        "Total user count",              # semantic
        "Top selling products",          # semantic
        "Something completely unrelated",  # no_match
    ]
    for q in queries:
        await medha.search(q)

    s = await medha.stats()

    print(f"{'Strategy':<20} {'Count':>6} {'Avg ms':>8}")
    print("-" * 38)
    for name, st in sorted(s.by_strategy.items()):
        print(f"{name:<20} {st.count:>6} {st.avg_latency_ms:>8.1f}")

## Cell 3: Latency Percentiles — p50 / p95 / p99

Percentiles are computed from the last `stats_max_latency_samples` searches
(default 10 000). They let you spot tail-latency spikes that the average hides.

In [ ]:
settings = Settings(backend_type="memory", stats_max_latency_samples=10_000)

async with Medha("obs_latency", embedder=embedder, settings=settings) as medha:
    entries = [
        {"question": f"Question number {i}", "generated_query": f"SELECT {i} FROM t"}
        for i in range(20)
    ]
    await medha.store_many(entries)

    for i in range(50):
        await medha.search(f"Question number {i % 25}")

    s = await medha.stats()
    print(f"Requests : {s.total_requests}")
    print(f"p50      : {s.p50_latency_ms:.2f} ms")
    print(f"p95      : {s.p95_latency_ms:.2f} ms")
    print(f"p99      : {s.p99_latency_ms:.2f} ms")
    print(f"avg      : {s.avg_latency_ms:.2f} ms")

## Cell 4: `reset_stats()` — Slide the Measurement Window

Call `reset_stats()` before a benchmarking run to discard warm-up traffic.
The `since` / `until` timestamps on `CacheStats` tell you the window covered.

In [ ]:
settings = Settings(backend_type="memory")

async with Medha("obs_reset", embedder=embedder, settings=settings) as medha:
    await medha.store("User count", "SELECT COUNT(*) FROM users")

    # Warm-up (discard)
    for _ in range(5):
        await medha.search("User count")

    before = await medha.stats()
    print(f"Before reset: requests={before.total_requests}, since={before.since.isoformat()}")

    await medha.reset_stats()

    # Benchmark window
    for _ in range(3):
        await medha.search("User count")

    after = await medha.stats()
    print(f"After reset : requests={after.total_requests}, since={after.since.isoformat()}")
    assert after.total_requests == 3

## Cell 5: `collect_stats=False` — Disable Overhead

Stats collection acquires an `asyncio.Lock` on every `search()` call.
For very high-throughput deployments where the overhead matters, disable it.

In [ ]:
settings = Settings(backend_type="memory", collect_stats=False)

async with Medha("obs_nostats", embedder=embedder, settings=settings) as medha:
    await medha.store("User count", "SELECT COUNT(*) FROM users")
    await medha.search("User count")

    s = await medha.stats()
    print(f"total_requests : {s.total_requests}")
    print(f"total_hits     : {s.total_hits}")
    assert s.total_requests == 0, "Stats collection is disabled — counters stay at zero"

## Cell 6: Python Logging Integration

Medha logs under the `medha` logger name:

| Level   | What you see |
|---------|-------------|
| `DEBUG` | Every tier decision: L1 hit/miss, template score, embedding cache key |
| `INFO`  | `store()` confirmations, batch counts, `start()`/`close()` |
| `WARNING` | Empty questions, oversized inputs, legacy template collections |
| `ERROR` | Embedding failures, backend errors |

In [ ]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
# Silence noisy third-party loggers
for lib in ("httpx", "httpcore", "fastembed", "hpack"):
    logging.getLogger(lib).setLevel(logging.WARNING)

settings = Settings(backend_type="memory")

async with Medha("obs_logging", embedder=embedder, settings=settings) as medha:
    await medha.store("User count", "SELECT COUNT(*) FROM users")
    hit = await medha.search("How many users are there?")
    print(f"\nResult: strategy={hit.strategy.value}, confidence={hit.confidence:.3f}")

## Cell 7: Capturing Logs Programmatically

For testing or audit pipelines, attach a custom handler to harvest log records
without writing to the console.

In [ ]:
import logging

logging.getLogger("medha").setLevel(logging.INFO)  # back to INFO after Cell 6

captured: list[logging.LogRecord] = []

class ListHandler(logging.Handler):
    def emit(self, record: logging.LogRecord) -> None:
        captured.append(record)

handler = ListHandler()
logging.getLogger("medha").addHandler(handler)

settings = Settings(backend_type="memory")

async with Medha("obs_capture", embedder=embedder, settings=settings) as medha:
    await medha.store("Revenue", "SELECT SUM(amount) FROM orders")
    await medha.search("Total revenue")

logging.getLogger("medha").removeHandler(handler)

print(f"Captured {len(captured)} INFO+ log records:")
for r in captured:
    print(f"  [{r.levelname}] {r.getMessage()}")

## Pattern: Production Observability Setup

```python
import logging
from medha import Medha, Settings

logging.getLogger("medha").setLevel(logging.INFO)

settings = Settings(
    backend_type="qdrant",
    collect_stats=True,
    stats_max_latency_samples=50_000,
)

async with Medha("prod_cache", embedder=embedder, settings=settings) as medha:
    # ... serve traffic ...

    # Emit a periodic stats snapshot (e.g. every 1 000 requests or every minute)
    s = await medha.stats()
    logging.getLogger("medha").info(
        "cache_stats hit_rate=%.3f p95_ms=%.1f backend_count=%d",
        s.hit_rate, s.p95_latency_ms, s.backend_count,
    )
    await medha.reset_stats()  # slide the measurement window
```